In [1]:
# Inject library so it is import-able
import sys
from pathlib import Path

project_directory = Path.cwd()
source_directory = project_directory / "src"

if not (source_directory / "data_quality" / "__init__.py").exists():
    raise FileNotFoundError(
        f"Could not find the data_quality source code under {source_directory}"
    )
if str(source_directory) not in sys.path:
    sys.path.insert(0, str(source_directory))

---
### pandas

##### General use

In [2]:
from __future__ import annotations

import json
from decimal import Decimal

import pandas as pandas_module

from data_quality import (
    AmbiguousBackendError,
    BackendDetectorError,
    BackendName,
    BackendRegistry,
    DEFAULT_BACKEND_REGISTRY,
    DEFAULT_PHYSICAL_SCHEMA_REGISTRY,
    DuplicateDetectorNameError,
    DuplicateSchemaDiscovererError,
    MissingDataFrameError,
    PhysicalSchemaDiscoveryError,
    PhysicalSchemaRegistry,
    RowCountMode,
    UnsupportedDataFrameError,
    UnsupportedSchemaDiscoveryBackendError,
    discover_physical_schema,
    discover_physical_schema_from_input,
    identify_backend,
    resolve_dataframe_input,
)

sample_dataframe = pandas_module.concat(
    [
        pandas_module.Series(
            [1001, 1002, pandas_module.NA, 1004, 1005],
            dtype = "Int64",
            name = "customer_id",
        ),
        pandas_module.Series(
            [
                Decimal("12000.50"),
                Decimal("8750.00"),
                None,
                Decimal("22100.75"),
                Decimal("5400.00"),
            ],
            dtype = "object",
            name = "account_balance_decimal",
        ),
        pandas_module.Series(
            [12000.50, 8750.00, pandas_module.NA, 22100.75, 5400.00],
            dtype="Float64",
            name="account_balance_float",
        ),
        pandas_module.Series(
            [True, False, pandas_module.NA, True, False],
            dtype = "boolean",
            name = "is_active",
        ),
        pandas_module.Series(
            [
                "alice@example.com",
                "bob@example.com",
                pandas_module.NA,
                "diana@example.com",
                "evan@example.com",
            ],
            dtype = pandas_module.StringDtype(storage="python"),
            name = "email",
        ),
        pandas_module.Series(
            pandas_module.Categorical(
                ["low", "medium", "high", "medium", "low"],
                categories = ["low", "medium", "high"],
                ordered = True,
            ),
            name = "risk_band",
        ),
        pandas_module.Series(
            pandas_module.to_datetime(
                [
                    "2026-08-18",
                    "2026-08-19",
                    "2026-08-20",
                    "2026-08-21",
                    "2026-08-22",
                ]
            ),
            name = "application_date",
        ),
        pandas_module.Series(
            pandas_module.to_datetime(
                [
                    "2026-08-18T01:15:00Z",
                    "2026-08-19T02:30:00Z",
                    "2026-08-20T03:45:00Z",
                    "2026-08-21T04:00:00Z",
                    "2026-08-22T05:10:00Z",
                ],
                utc = True,
            ).tz_convert(
                "Australia/Sydney"
            ),
            name = "event_timestamp",
        ),
        pandas_module.Series(
            pandas_module.to_timedelta(["5m", "7m", None, "4m", "10m"]),
            name = "processing_duration",
        ),
        pandas_module.Series(
            pandas_module.period_range("2026-04", periods = 5, freq = "M"),
            name = "reporting_period",
        ),
        pandas_module.Series(
            [
                {"source": "web"},
                {"source": "dealer"},
                None,
                {"source": "broker"},
                {"source": "web"},
            ],
            dtype = "object",
            name = ("metadata", "payload"),
        ),
        pandas_module.Series(
            ["new", "approved", "pending", "approved", "declined"],
            dtype = "string",
            name = "status",
        ),
        pandas_module.Series(
            pandas_module.Categorical(
                ["online", "dealer", "broker", "dealer", "online"]
            ),
            name = "status",
        ),
        pandas_module.Series(
            [1, 2, 3, 4, 5],
            dtype = "int64",
            name = 2026,
        ),
    ],
    axis = 1,
)

# Backend identification
backend_identity = identify_backend(sample_dataframe)
print("BACKEND")
print(json.dumps(backend_identity.as_dict(), indent = 2))
# Retain the original object plus its identity
dataframe_input = resolve_dataframe_input(sample_dataframe)
assert dataframe_input.dataframe is sample_dataframe
assert dataframe_input.backend == backend_identity
assert "alice@example.com" not in repr(dataframe_input)
# Avoids repeating backend detection
physical_schema = discover_physical_schema_from_input(
    dataframe_input,
    row_count_mode=RowCountMode.METADATA_ONLY,
)
assert physical_schema.backend is dataframe_input.backend
print("\nDIMENSIONS")
print("shape:", physical_schema.shape)
print("row count:", physical_schema.dimensions.row_count)
print("column count:", physical_schema.dimensions.column_count)
print("row-count source:", physical_schema.dimensions.row_count_source.value)
print("has exact row count:", physical_schema.dimensions.has_exact_row_count)

print("\nCOLUMNS")
for column in physical_schema.columns:
    print(
        {
            "position": column.position,
            "exact_name_object": column.name,
            "report_name": column.name_string,
            "native_dtype_object": column.native_data_type,
            "native_dtype_string": column.native_data_type_string,
            "native_dtype_class": column.native_data_type_class,
            "nullable_schema_metadata": column.nullable,
        }
    )

# The native pandas schema is the `Series`` supplied by `DataFrame.dtypes`
assert isinstance(physical_schema.native_schema, pandas_module.Series)
assert physical_schema.native_schema.equals(sample_dataframe.dtypes)
print("\nNATIVE PANDAS SCHEMA")
print(physical_schema.native_schema)

# Position preserves duplicate labels and exact non-string label objects
assert [column.position for column in physical_schema.columns] == list(
    range(sample_dataframe.shape[1])
)
assert sum(column.name == "status" for column in physical_schema.columns) == 2
assert physical_schema.columns[-1].name == 2026
assert isinstance(physical_schema.columns[-1].name, int)

# `as_dict` strips backend-native objects and is JSON serialisable
report_dictionary = physical_schema.as_dict()
report_json = json.dumps(report_dictionary, indent = 2)
print("\nJSON-SAFE REPORT")
print(report_json)

# The report-friendly column records can themselves be viewed as a dataframe
schema_table = pandas_module.DataFrame(report_dictionary["columns"])
print("\nSCHEMA TABLE")
print(schema_table.to_string(index=False))

# The convenience API performs all 3 implemented stages above in one call
convenience_schema = discover_physical_schema(sample_dataframe)
assert convenience_schema.as_dict() == physical_schema.as_dict()

# For pandas, both row-count modes use DataFrame.shape metadata
# `EXACT` does not run a separate count query and therefore has the same result/source
metadata_only_schema = discover_physical_schema(
    sample_dataframe,
    row_count_mode=RowCountMode.METADATA_ONLY,
)
exact_schema = discover_physical_schema(
    sample_dataframe,
    row_count_mode=RowCountMode.EXACT,
)
assert metadata_only_schema.shape == exact_schema.shape == sample_dataframe.shape
assert metadata_only_schema.dimensions.row_count_source.value == "frame_metadata"
assert exact_schema.dimensions.row_count_source.value == "frame_metadata"

# `repr` safety
# Native dataframe values and the native dtype Series are excluded from representations intended for ordinary diagnostics/logging
assert "alice@example.com" not in repr(dataframe_input)
assert "alice@example.com" not in repr(physical_schema)

# Pandas DataFrame subclasses are also detected as pandas
class CustomPandasDataFrame(pandas_module.DataFrame):
    pass
custom_dataframe = CustomPandasDataFrame({"identifier": [1, 2]})
custom_backend = identify_backend(custom_dataframe)
assert custom_backend.name is BackendName.PANDAS
assert custom_backend.dataframe_type.endswith("CustomPandasDataFrame")

# Empty dataframes have a finite, exact (0, 0) shape
empty_schema = discover_physical_schema(pandas_module.DataFrame())
assert empty_schema.shape == (0, 0)
assert empty_schema.columns == ()

BACKEND
{
  "name": "pandas",
  "dataframe_api": "pandas",
  "frame_kind": "dataframe",
  "evaluation_mode": "eager",
  "distribution_mode": "local",
  "connection_mode": "in_process",
  "dataframe_type": "pandas.core.frame.DataFrame",
  "client_library_version": "2.3.3"
}

DIMENSIONS
shape: (5, 14)
row count: 5
column count: 14
row-count source: frame_metadata
has exact row count: True

COLUMNS
{'position': 0, 'exact_name_object': 'customer_id', 'report_name': 'customer_id', 'native_dtype_object': Int64Dtype(), 'native_dtype_string': 'Int64', 'native_dtype_class': 'pandas.core.arrays.integer.Int64Dtype', 'nullable_schema_metadata': None}
{'position': 1, 'exact_name_object': 'account_balance_decimal', 'report_name': 'account_balance_decimal', 'native_dtype_object': dtype('O'), 'native_dtype_string': 'object', 'native_dtype_class': 'numpy.dtypes.ObjectDType', 'nullable_schema_metadata': None}
{'position': 2, 'exact_name_object': 'account_balance_float', 'report_name': 'account_balance_f

##### Exceptions

In [3]:
# Expected public errors can be caught at the appropriate boundary
try:
    identify_backend(None)
except MissingDataFrameError as error:
    print("\nEXPECTED MISSING INPUT ERROR:", error)

try:
    identify_backend(pandas_module.Series([1, 2, 3]))
except UnsupportedDataFrameError as error:
    print("EXPECTED UNSUPPORTED INPUT ERROR:", error)

try:
    discover_physical_schema_from_input(sample_dataframe)
except TypeError as error:
    print("EXPECTED RAW-INPUT ERROR:", error)

try:
    discover_physical_schema_from_input(
        dataframe_input,
        row_count_mode = "exact"
    )
except TypeError as error:
    print("EXPECTED ROW-COUNT MODE ERROR:", error)

# Advanced extension point, registry extension is immutable
class NeverMatchingDetector:
    name = "example-never-matching"
    supported_inputs = ("example.NeverMatchingDataFrame",)

    def detect(self, 
               dataframe: object):

        return None
extended_backend_registry = DEFAULT_BACKEND_REGISTRY.with_detector(
    NeverMatchingDetector()
)
assert len(extended_backend_registry.detectors) == (
    len(DEFAULT_BACKEND_REGISTRY.detectors) + 1
)
assert len(DEFAULT_BACKEND_REGISTRY.detectors) == 3
assert identify_backend(
    sample_dataframe,
    registry = extended_backend_registry,
).name is BackendName.PANDAS
print("\nSUPPORTED INPUTS")
print(DEFAULT_BACKEND_REGISTRY.supported_inputs)

# Advanced extension point, inject a replacement schema registry for this call without mutating the process-wide default registry
default_pandas_discoverer = next(
    discoverer
    for discoverer in DEFAULT_PHYSICAL_SCHEMA_REGISTRY.discoverers
    if discoverer.backend_name is BackendName.PANDAS
)
class RecordingPandasSchemaDiscoverer:
    backend_name = BackendName.PANDAS

    def __init__(self, 
                 delegate: object) -> None:
        self.delegate = delegate
        self.call_count = 0

    def discover(self, 
                 dataframe, 
                 backend, 
                 row_count_mode):
        self.call_count += 1

        return self.delegate.discover(dataframe, backend, row_count_mode)
recording_discoverer = RecordingPandasSchemaDiscoverer(
    default_pandas_discoverer
)
empty_schema_registry = PhysicalSchemaRegistry(())
recording_schema_registry = empty_schema_registry.with_discoverer(
    recording_discoverer
)
assert empty_schema_registry.discoverers == ()
recorded_schema = discover_physical_schema(
    sample_dataframe,
    schema_registry = recording_schema_registry,
)
assert recorded_schema.shape == sample_dataframe.shape
assert recording_discoverer.call_count == 1
assert DEFAULT_PHYSICAL_SCHEMA_REGISTRY.discoverers[0] is default_pandas_discoverer

# Registry guardrails
try:
    BackendRegistry((NeverMatchingDetector(), NeverMatchingDetector()))
except DuplicateDetectorNameError as error:
    print("EXPECTED DUPLICATE DETECTOR ERROR:", error)

try:
    PhysicalSchemaRegistry(
        (default_pandas_discoverer, default_pandas_discoverer)
    )
except DuplicateSchemaDiscovererError as error:
    print("EXPECTED DUPLICATE DISCOVERER ERROR:", error)

class ConflictingPandasDetector:
    name = "conflicting-pandas"
    supported_inputs = ("pandas.DataFrame",)

    def detect(self, 
               dataframe: object):
        if isinstance(dataframe, pandas_module.DataFrame):

            return backend_identity
        
        return None
ambiguous_registry = DEFAULT_BACKEND_REGISTRY.with_detector(
    ConflictingPandasDetector()
)
try:
    identify_backend(sample_dataframe, registry = ambiguous_registry)
except AmbiguousBackendError as error:
    print("EXPECTED AMBIGUOUS BACKEND ERROR:", error)

class FailingBackendDetector:
    name = "failing"
    supported_inputs = ("pandas.DataFrame",)

    def detect(self, 
               dataframe: object):
        raise RuntimeError("Underlying detector failure")
try:
    identify_backend(
        sample_dataframe,
        registry = BackendRegistry((FailingBackendDetector(),)),
    )
except BackendDetectorError as error:
    assert isinstance(error.__cause__, RuntimeError)
    print("EXPECTED WRAPPED DETECTOR ERROR:", error)

try:
    discover_physical_schema_from_input(
        dataframe_input,
        registry = PhysicalSchemaRegistry(()),
    )
except UnsupportedSchemaDiscoveryBackendError as error:
    print("EXPECTED MISSING DISCOVERER ERROR:", error)

class FailingPandasSchemaDiscoverer:
    backend_name = BackendName.PANDAS

    def discover(self, 
                 dataframe, 
                 backend, 
                 row_count_mode):
        raise RuntimeError("Underlying schema-discovery failure")
try:
    discover_physical_schema_from_input(
        dataframe_input,
        registry = PhysicalSchemaRegistry((FailingPandasSchemaDiscoverer(),))
    )
except PhysicalSchemaDiscoveryError as error:
    assert isinstance(error.__cause__, RuntimeError)
    print("EXPECTED WRAPPED SCHEMA ERROR:", error)


EXPECTED MISSING INPUT ERROR: Expected a dataframe but received `None`
EXPECTED UNSUPPORTED INPUT ERROR: Unsupported dataframe type 'pandas.core.series.Series', supported inputs are: pandas.DataFrame, polars.DataFrame, polars.LazyFrame, pyspark.sql.DataFrame (Spark classic), pyspark.sql.DataFrame (Spark Connect)
EXPECTED RAW-INPUT ERROR: Expected `DataFrameInput`, call `resolve_dataframe_input()` first or pass the native dataframe to `discover_physical_schema()`
EXPECTED ROW-COUNT MODE ERROR: row_count_mode must be a RowCountMode value.

SUPPORTED INPUTS
('pandas.DataFrame', 'polars.DataFrame', 'polars.LazyFrame', 'pyspark.sql.DataFrame (Spark classic)', 'pyspark.sql.DataFrame (Spark Connect)')
EXPECTED DUPLICATE DETECTOR ERROR: A backend detector named 'example-never-matching' is already registered
EXPECTED DUPLICATE DISCOVERER ERROR: A physical-schema discoverer for backend 'pandas' is already registered
EXPECTED AMBIGUOUS BACKEND ERROR: Backend detection is ambiguous for 'pandas.co

---
### Polars

##### General use - eager frame

In [4]:
from __future__ import annotations

import json
from datetime import date, datetime, timedelta
from decimal import Decimal
from zoneinfo import ZoneInfo

import polars as polars_module

from data_quality import (
    AmbiguousBackendError,
    BackendDetectorError,
    BackendIdentity,
    BackendName,
    BackendRegistry,
    DEFAULT_BACKEND_REGISTRY,
    DEFAULT_PHYSICAL_SCHEMA_REGISTRY,
    DataFramePhysicalSchema,
    DuplicateDetectorNameError,
    DuplicateSchemaDiscovererError,
    EvaluationMode,
    FrameKind,
    MissingDataFrameError,
    PhysicalSchemaDiscoverer,
    PhysicalSchemaDiscoveryError,
    PhysicalSchemaRegistry,
    RowCountMode,
    RowCountSource,
    UnsupportedDataFrameError,
    UnsupportedSchemaDiscoveryBackendError,
    discover_physical_schema,
    discover_physical_schema_from_input,
    identify_backend,
    resolve_dataframe_input,
)

sydney_time_zone = ZoneInfo("Australia/Sydney")
sample_dataframe = polars_module.DataFrame(
    [
        polars_module.Series(
            "customer_identifier",
            [1001, 1002, None, 1004, 1005],
            dtype=polars_module.Int64,
        ),
        polars_module.Series(
            "application_sequence",
            [1, 2, 3, 4, 5],
            dtype=polars_module.UInt32,
        ),
        polars_module.Series(
            "account_balance",
            [
                Decimal("12000.50"),
                Decimal("8750.00"),
                None,
                Decimal("22100.75"),
                Decimal("5400.00"),
            ],
            dtype=polars_module.Decimal(precision=12, scale=2),
        ),
        polars_module.Series(
            "risk_score",
            [0.12, 0.36, None, 0.81, 0.44],
            dtype=polars_module.Float64,
        ),
        polars_module.Series(
            "is_active",
            [True, False, None, True, False],
            dtype=polars_module.Boolean,
        ),
        polars_module.Series(
            "email_address",
            [
                "alice@example.com",
                "bob@example.com",
                None,
                "diana@example.com",
                "evan@example.com",
            ],
            dtype=polars_module.String,
        ),
        polars_module.Series(
            "risk_band",
            ["low", "medium", "high", "medium", "low"],
            dtype=polars_module.Categorical,
        ),
        polars_module.Series(
            "application_date",
            [
                date(2026, 8, 18),
                date(2026, 8, 19),
                date(2026, 8, 20),
                date(2026, 8, 21),
                date(2026, 8, 22),
            ],
            dtype=polars_module.Date,
        ),
        polars_module.Series(
            "event_timestamp",
            [
                datetime(2026, 8, 18, 9, 15, tzinfo=sydney_time_zone),
                datetime(2026, 8, 19, 10, 30, tzinfo=sydney_time_zone),
                datetime(2026, 8, 20, 11, 45, tzinfo=sydney_time_zone),
                datetime(2026, 8, 21, 12, 0, tzinfo=sydney_time_zone),
                datetime(2026, 8, 22, 13, 10, tzinfo=sydney_time_zone),
            ],
            dtype=polars_module.Datetime(
                time_unit="us",
                time_zone="Australia/Sydney",
            ),
        ),
        polars_module.Series(
            "processing_duration",
            [
                timedelta(minutes=5),
                timedelta(minutes=7),
                None,
                timedelta(minutes=4),
                timedelta(minutes=10),
            ],
            dtype=polars_module.Duration(time_unit="us"),
        ),
        polars_module.Series(
            "document_tags",
            [
                ["identity", "income"],
                ["identity"],
                None,
                ["income", "vehicle"],
                [],
            ],
            dtype=polars_module.List(polars_module.String),
        ),
        polars_module.Series(
            "metadata",
            [
                {"source": "web", "priority": 1},
                {"source": "dealer", "priority": 2},
                None,
                {"source": "broker", "priority": 3},
                {"source": "web", "priority": 1},
            ],
            dtype=polars_module.Struct(
                {
                    "source": polars_module.String,
                    "priority": polars_module.Int64,
                }
            ),
        ),
        polars_module.Series(
            "payload_binary",
            [b"A", b"B", None, b"D", b"E"],
            dtype=polars_module.Binary,
        ),
    ]
)

def print_physical_schema(title: str,
                          physical_schema: DataFramePhysicalSchema) -> None:
    print(f"\n{title}")
    print("backend:", physical_schema.backend.as_dict())
    print("dimensions:", physical_schema.dimensions.as_dict())

    for column in physical_schema.columns:
        print(
            {
                "position": column.position,
                "exact_name_object": column.name,
                "report_name": column.name_string,
                "native_dtype_object": column.native_data_type,
                "native_dtype_string": column.native_data_type_string,
                "native_dtype_class": column.native_data_type_class,
                "nullable_schema_metadata": column.nullable,
            }
        )

# Eager Polars DataFrame
eager_backend_identity = identify_backend(sample_dataframe)
assert eager_backend_identity.name is BackendName.POLARS
assert eager_backend_identity.frame_kind is FrameKind.DATAFRAME
assert eager_backend_identity.evaluation_mode is EvaluationMode.EAGER
print("EAGER BACKEND")
print(json.dumps(eager_backend_identity.as_dict(), indent = 2))
# Preserve the exact user-supplied frame
eager_dataframe_input = resolve_dataframe_input(sample_dataframe)
assert eager_dataframe_input.dataframe is sample_dataframe
assert eager_dataframe_input.backend == eager_backend_identity
assert "alice@example.com" not in repr(eager_dataframe_input)
eager_physical_schema = discover_physical_schema_from_input(
    eager_dataframe_input,
    row_count_mode = RowCountMode.METADATA_ONLY,
)
assert eager_physical_schema.backend is eager_dataframe_input.backend
assert eager_physical_schema.shape == sample_dataframe.shape
assert eager_physical_schema.dimensions.row_count_source is (
    RowCountSource.FRAME_METADATA
)
assert eager_physical_schema.dimensions.has_exact_row_count
print_physical_schema("EAGER PHYSICAL SCHEMA", eager_physical_schema)
# For eager Polars, `native_schema` is the ordered schema mapping returned by `DataFrame.schema`
assert eager_physical_schema.native_schema == sample_dataframe.schema
assert tuple(eager_physical_schema.native_schema.items()) == tuple(
    sample_dataframe.schema.items()
)
# Polars column names are strings and its schema mapping is unique by name
assert all(
    isinstance(column.name, str)
    for column in eager_physical_schema.columns
)
assert len({column.name for column in eager_physical_schema.columns}) == len(
    eager_physical_schema.columns
)
# Every exact native dtype object is retained in original column order
for column, (native_name, native_dtype) in zip(
    eager_physical_schema.columns,
    sample_dataframe.schema.items(),
    strict = True,
):
    assert column.name == native_name
    assert column.native_data_type == native_dtype

# `as_dict` contains primitives rather than Polars `Schema`/`DataType` objects
eager_report_dictionary = eager_physical_schema.as_dict()
eager_report_json = json.dumps(eager_report_dictionary, indent=2)
print("\nEAGER JSON-SAFE REPORT")
print(eager_report_json)

eager_schema_table = polars_module.DataFrame(
    eager_report_dictionary["columns"]
)
print("\nEAGER REPORT TABLE")
print(eager_schema_table)

eager_convenience_schema = discover_physical_schema(sample_dataframe)
assert eager_convenience_schema.as_dict() == eager_physical_schema.as_dict()

# Both modes use DataFrame.shape for an eager Polars frame 
# `EXACT` does not run a query because an exact finite height is already frame metadata
eager_metadata_schema = discover_physical_schema(
    sample_dataframe,
    row_count_mode = RowCountMode.METADATA_ONLY,
)
eager_exact_schema = discover_physical_schema(
    sample_dataframe,
    row_count_mode = RowCountMode.EXACT,
)
assert eager_metadata_schema.shape == eager_exact_schema.shape
assert eager_exact_schema.shape == sample_dataframe.shape
assert eager_metadata_schema.dimensions.row_count_source is (
    RowCountSource.FRAME_METADATA
)
assert eager_exact_schema.dimensions.row_count_source is (
    RowCountSource.FRAME_METADATA
)
# Values are excluded from the input/schema representations
assert "alice@example.com" not in repr(eager_dataframe_input)
assert "alice@example.com" not in repr(eager_physical_schema)

# Empty frame validation
empty_eager_schema = discover_physical_schema(polars_module.DataFrame())
assert empty_eager_schema.shape == (0, 0)
assert empty_eager_schema.columns == ()






try:
    identify_backend(None)
except MissingDataFrameError as error:
    print("\nEXPECTED MISSING INPUT ERROR:", error)

try:
    identify_backend(polars_module.Series("identifier", [1, 2, 3]))
except UnsupportedDataFrameError as error:
    print("EXPECTED UNSUPPORTED INPUT ERROR:", error)

try:
    discover_physical_schema_from_input(sample_dataframe)  # type: ignore[arg-type]
except TypeError as error:
    print("EXPECTED RAW-INPUT ERROR:", error)

try:
    discover_physical_schema_from_input(
        eager_dataframe_input,
        row_count_mode="exact",  # type: ignore[arg-type]
    )
except TypeError as error:
    print("EXPECTED ROW-COUNT MODE ERROR:", error)

EAGER BACKEND
{
  "name": "polars",
  "dataframe_api": "polars",
  "frame_kind": "dataframe",
  "evaluation_mode": "eager",
  "distribution_mode": "local",
  "connection_mode": "in_process",
  "dataframe_type": "polars.dataframe.frame.DataFrame",
  "client_library_version": "1.43.2"
}

EAGER PHYSICAL SCHEMA
backend: {'name': 'polars', 'dataframe_api': 'polars', 'frame_kind': 'dataframe', 'evaluation_mode': 'eager', 'distribution_mode': 'local', 'connection_mode': 'in_process', 'dataframe_type': 'polars.dataframe.frame.DataFrame', 'client_library_version': '1.43.2'}
dimensions: {'row_count': 5, 'column_count': 13, 'shape': [5, 13], 'row_count_source': 'frame_metadata', 'has_exact_row_count': True}
{'position': 0, 'exact_name_object': 'customer_identifier', 'report_name': 'customer_identifier', 'native_dtype_object': Int64, 'native_dtype_string': 'Int64', 'native_dtype_class': 'polars.datatypes.classes.Int64', 'nullable_schema_metadata': None}
{'position': 1, 'exact_name_object': 'applic

##### General use - lazy frame

In [5]:
# Create a lazy query plan from the in-memory eager frame, as follows: 
    # - The filter removes the one row whose customer identifier is null
    # - `with_columns`` adds one derived Float64 column 
# Neither operation is executed at construction time
sample_lazy_frame = (
    sample_dataframe.lazy()
    .filter(
        polars_module.col("customer_identifier").is_not_null()
    )
    .with_columns(
        (polars_module.col("risk_score") * polars_module.lit(100.0)).alias("risk_score_percentage")
    )
)

lazy_backend_identity = identify_backend(sample_lazy_frame)
assert lazy_backend_identity.name is BackendName.POLARS
assert lazy_backend_identity.frame_kind is FrameKind.LAZY_FRAME
assert lazy_backend_identity.evaluation_mode is EvaluationMode.LAZY
print("\nLAZY BACKEND")
print(json.dumps(lazy_backend_identity.as_dict(), indent = 2))

lazy_dataframe_input = resolve_dataframe_input(sample_lazy_frame)
assert lazy_dataframe_input.dataframe is sample_lazy_frame
# Metadata-only mode resolves the schema with `collect_schema()`, 
# But does not collect user rows or execute a row-count query
lazy_metadata_schema = discover_physical_schema_from_input(
    lazy_dataframe_input,
    row_count_mode = RowCountMode.METADATA_ONLY,
)
assert lazy_metadata_schema.shape == (None, sample_dataframe.width + 1)
assert lazy_metadata_schema.dimensions.row_count is None
assert not lazy_metadata_schema.dimensions.has_exact_row_count
assert lazy_metadata_schema.dimensions.row_count_source is (
    RowCountSource.NOT_COMPUTED
)
assert lazy_metadata_schema.native_schema == sample_lazy_frame.collect_schema()
print_physical_schema(
    "LAZY METADATA-ONLY PHYSICAL SCHEMA",
    lazy_metadata_schema,
)

# `EXACT` deliberately executes a scalar `pl.len() query`
# It does not return the user rows, only the exact row-count scalar is materialiSed
lazy_exact_schema = discover_physical_schema(
    sample_lazy_frame,
    row_count_mode = RowCountMode.EXACT,
)
assert lazy_exact_schema.shape == (4, sample_dataframe.width + 1)
assert lazy_exact_schema.dimensions.row_count == 4
assert lazy_exact_schema.dimensions.has_exact_row_count
assert lazy_exact_schema.dimensions.row_count_source is (
    RowCountSource.EXECUTED_QUERY
)
print_physical_schema("LAZY EXACT PHYSICAL SCHEMA", lazy_exact_schema)
# Both lazy results are JSON serialiSable
# Native `Schema` and `DataType` objects are retained outside the report-friendly dictionary
json.dumps(lazy_metadata_schema.as_dict())
json.dumps(lazy_exact_schema.as_dict())

# Empty frame validation
empty_lazy_schema = discover_physical_schema(
    polars_module.DataFrame().lazy(),
    row_count_mode = RowCountMode.EXACT,
)
assert empty_lazy_schema.shape == (0, 0)
assert empty_lazy_schema.columns == ()


LAZY BACKEND
{
  "name": "polars",
  "dataframe_api": "polars",
  "frame_kind": "lazy_frame",
  "evaluation_mode": "lazy",
  "distribution_mode": "local",
  "connection_mode": "in_process",
  "dataframe_type": "polars.lazyframe.frame.LazyFrame",
  "client_library_version": "1.43.2"
}

LAZY METADATA-ONLY PHYSICAL SCHEMA
backend: {'name': 'polars', 'dataframe_api': 'polars', 'frame_kind': 'lazy_frame', 'evaluation_mode': 'lazy', 'distribution_mode': 'local', 'connection_mode': 'in_process', 'dataframe_type': 'polars.lazyframe.frame.LazyFrame', 'client_library_version': '1.43.2'}
dimensions: {'row_count': None, 'column_count': 14, 'shape': [None, 14], 'row_count_source': 'not_computed', 'has_exact_row_count': False}
{'position': 0, 'exact_name_object': 'customer_identifier', 'report_name': 'customer_identifier', 'native_dtype_object': Int64, 'native_dtype_string': 'Int64', 'native_dtype_class': 'polars.datatypes.classes.Int64', 'nullable_schema_metadata': None}
{'position': 1, 'exact_nam

##### Exceptions

In [6]:
# Backend detector
class NeverMatchingDetector:
    name = "example-never-matching"
    supported_inputs = ("example.NeverMatchingDataFrame",)

    def detect(self, 
               dataframe: object) -> BackendIdentity | None:

        return None
extended_backend_registry = DEFAULT_BACKEND_REGISTRY.with_detector(
    NeverMatchingDetector()
)
assert len(extended_backend_registry.detectors) == (
    len(DEFAULT_BACKEND_REGISTRY.detectors) + 1
)
assert len(DEFAULT_BACKEND_REGISTRY.detectors) == 3
assert identify_backend(
    sample_dataframe,
    registry = extended_backend_registry,
).name is BackendName.POLARS
print("\nSUPPORTED INPUTS")
print(DEFAULT_BACKEND_REGISTRY.supported_inputs)

# Schema discoverer
default_polars_discoverer = next(
    discoverer
    for discoverer in DEFAULT_PHYSICAL_SCHEMA_REGISTRY.discoverers
    if discoverer.backend_name is BackendName.POLARS
)
class RecordingPolarsSchemaDiscoverer:
    backend_name = BackendName.POLARS

    def __init__(self, 
                 delegate: PhysicalSchemaDiscoverer) -> None:
        self.delegate = delegate
        self.call_count = 0

    def discover(self,
                 dataframe: object,
                 backend: BackendIdentity,
                 row_count_mode: RowCountMode) -> DataFramePhysicalSchema:
        self.call_count += 1

        return self.delegate.discover(
            dataframe, 
            backend, 
            row_count_mode
        )
recording_discoverer = RecordingPolarsSchemaDiscoverer(
    default_polars_discoverer
)
empty_schema_registry = PhysicalSchemaRegistry(())
recording_schema_registry = empty_schema_registry.with_discoverer(
    recording_discoverer
)
assert empty_schema_registry.discoverers == ()

recorded_schema = discover_physical_schema(
    sample_dataframe,
    schema_registry = recording_schema_registry,
)
assert recorded_schema.shape == sample_dataframe.shape
assert recording_discoverer.call_count == 1

try:
    BackendRegistry((NeverMatchingDetector(), NeverMatchingDetector()))
except DuplicateDetectorNameError as error:
    print("EXPECTED DUPLICATE DETECTOR ERROR:", error)

try:
    PhysicalSchemaRegistry(
        (default_polars_discoverer, default_polars_discoverer)
    )
except DuplicateSchemaDiscovererError as error:
    print("EXPECTED DUPLICATE DISCOVERER ERROR:", error)

# Conflicting backend
class ConflictingPolarsDetector:
    name = "conflicting-polars"
    supported_inputs = ("polars.DataFrame", "polars.LazyFrame")

    def detect(self, 
               dataframe: object) -> BackendIdentity | None:
        if dataframe is sample_dataframe:

            return eager_backend_identity

        return None
ambiguous_registry = DEFAULT_BACKEND_REGISTRY.with_detector(
    ConflictingPolarsDetector()
)
try:
    identify_backend(sample_dataframe, registry = ambiguous_registry)
except AmbiguousBackendError as error:
    print("EXPECTED AMBIGUOUS BACKEND ERROR:", error)

class FailingBackendDetector:
    name = "failing"
    supported_inputs = ("polars.DataFrame",)

    def detect(self, 
               dataframe: object) -> BackendIdentity | None:
        raise RuntimeError("underlying detector failure")
try:
    identify_backend(
        sample_dataframe,
        registry = BackendRegistry((FailingBackendDetector(),)),
    )
except BackendDetectorError as error:
    assert isinstance(error.__cause__, RuntimeError)
    print("EXPECTED WRAPPED DETECTOR ERROR:", error)

try:
    discover_physical_schema_from_input(
        eager_dataframe_input,
        registry = PhysicalSchemaRegistry(()),
    )
except UnsupportedSchemaDiscoveryBackendError as error:
    print("EXPECTED MISSING DISCOVERER ERROR:", error)

class FailingPolarsSchemaDiscoverer:
    backend_name = BackendName.POLARS

    def discover(self,
                 dataframe: object,
                 backend: BackendIdentity,
                 row_count_mode: RowCountMode) -> DataFramePhysicalSchema:
        raise RuntimeError("Underlying schema-discovery failure")
try:
    discover_physical_schema_from_input(
        eager_dataframe_input,
        registry = PhysicalSchemaRegistry((FailingPolarsSchemaDiscoverer(),)),
    )
except PhysicalSchemaDiscoveryError as error:
    assert isinstance(error.__cause__, RuntimeError)
    print("EXPECTED WRAPPED SCHEMA ERROR:", error)


SUPPORTED INPUTS
('pandas.DataFrame', 'polars.DataFrame', 'polars.LazyFrame', 'pyspark.sql.DataFrame (Spark classic)', 'pyspark.sql.DataFrame (Spark Connect)')
EXPECTED DUPLICATE DETECTOR ERROR: A backend detector named 'example-never-matching' is already registered
EXPECTED DUPLICATE DISCOVERER ERROR: A physical-schema discoverer for backend 'polars' is already registered
EXPECTED AMBIGUOUS BACKEND ERROR: Backend detection is ambiguous for 'polars.dataframe.frame.DataFrame' with matching detectors of polars, conflicting-polars
EXPECTED WRAPPED DETECTOR ERROR: Backend detector 'failing' failed while inspecting 'polars.dataframe.frame.DataFrame'
EXPECTED MISSING DISCOVERER ERROR: No physical-schema discoverer is registered for backend 'polars'
EXPECTED WRAPPED SCHEMA ERROR: Physical-schema discovery failed for backend 'polars' and dataframe type 'polars.dataframe.frame.DataFrame'


---
### PySpark

In [7]:
import os
import sys

import json
from datetime import date, datetime, timezone
from decimal import Decimal

from pyspark.sql import Row, SparkSession
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    DateType,
    DecimalType,
    DoubleType,
    LongType,
    MapType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

from data_quality import (
    AmbiguousBackendError,
    BackendDetectorError,
    BackendIdentity,
    BackendName,
    BackendRegistry,
    ConnectionMode,
    DEFAULT_BACKEND_REGISTRY,
    DEFAULT_PHYSICAL_SCHEMA_REGISTRY,
    DataFrameApi,
    DataFramePhysicalSchema,
    DistributionMode,
    DuplicateDetectorNameError,
    DuplicateSchemaDiscovererError,
    EvaluationMode,
    FrameKind,
    MissingDataFrameError,
    PhysicalSchemaDiscoverer,
    PhysicalSchemaDiscoveryError,
    PhysicalSchemaRegistry,
    RowCountMode,
    RowCountSource,
    UnsupportedDataFrameError,
    UnsupportedSchemaDiscoveryBackendError,
    discover_physical_schema,
    discover_physical_schema_from_input,
    identify_backend,
    resolve_dataframe_input,
)

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
spark = (
    SparkSession.builder
    .master("local[1]")
    .appName("example_notebook")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .config(
        "spark.sql.session.timeZone",
        "Australia/Sydney",
    )
    .getOrCreate()
)
print("Spark session started with version number", spark.version)

address_schema = StructType(
    [
        StructField(
            "suburb",
            StringType(),
            nullable = False,
        ),
        StructField(
            "postcode",
            StringType(),
            nullable = True,
        ),
    ]
)
event_schema = StructType(
    [
        StructField(
            "event_type",
            StringType(),
            nullable = False,
        ),
        StructField(
            "event_value",
            DoubleType(),
            nullable = True,
        ),
    ]
)
sample_schema = StructType(
    [
        StructField(
            "customer_id",
            LongType(),
            nullable = False,
            metadata = {
                "description": "Unique customer identifier",
                "primary_key_candidate": True,
            },
        ),
        StructField(
            "email_address",
            StringType(),
            nullable = True,
        ),
        StructField(
            "is_active",
            BooleanType(),
            nullable = True,
        ),
        StructField(
            "amount_financed",
            DecimalType(
                precision = 12,
                scale = 2,
            ),
            nullable = True,
        ),
        StructField(
            "risk_score",
            DoubleType(),
            nullable = True,
        ),
        StructField(
            "application_date",
            DateType(),
            nullable = True,
        ),
        StructField(
            "event_timestamp",
            TimestampType(),
            nullable = True,
        ),
        StructField(
            "tags",
            ArrayType(
                StringType(),
                containsNull = False,
            ),
            nullable = True,
        ),
        StructField(
            "attributes",
            MapType(
                keyType = StringType(),
                valueType = StringType(),
                valueContainsNull = True,
            ),
            nullable = True,
        ),
        StructField(
            "address",
            address_schema,
            nullable = True,
        ),
        StructField(
            "events",
            ArrayType(
                event_schema,
                containsNull = False,
            ),
            nullable = True,
        ),
    ]
)
sample_data = [
    {
        "customer_id": 1001,
        "email_address": "alice@example.com",
        "is_active": True,
        "amount_financed": Decimal("12000.50"),
        "risk_score": 0.93,
        "application_date": date(2026, 8, 18),
        "event_timestamp": datetime(
            2026,
            8,
            18,
            10,
            15,
            0,
        ),
        "tags": [
            "digital",
            "priority",
        ],
        "attributes": {
            "channel": "web",
            "campaign": None,
        },
        "address": {
            "suburb": "Sydney",
            "postcode": "2000",
        },
        "events": [
            {
                "event_type": "created",
                "event_value": 1.0,
            },
            {
                "event_type": "reviewed",
                "event_value": 0.75,
            },
        ],
    },
    {
        "customer_id": 1002,
        "email_address": "bob@example.com",
        "is_active": False,
        "amount_financed": Decimal("8750.00"),
        "risk_score": 0.64,
        "application_date": date(2026, 8, 19),
        "event_timestamp": datetime(
            2026,
            8,
            19,
            11,
            30,
            0,
        ),
        "tags": [
            "dealer",
        ],
        "attributes": {
            "channel": "dealer",
            "campaign": "winter",
        },
        "address": {
            "suburb": "Parramatta",
            "postcode": None,
        },
        "events": [],
    },
    {
        "customer_id": 1003,
        "email_address": None,
        "is_active": None,
        "amount_financed": None,
        "risk_score": None,
        "application_date": date(2026, 8, 20),
        "event_timestamp": None,
        "tags": None,
        "attributes": None,
        "address": None,
        "events": [
            {
                "event_type": "created",
                "event_value": None,
            },
        ],
    },
    {
        "customer_id": 1004,
        "email_address": "charlie@example.com",
        "is_active": True,
        "amount_financed": Decimal("15650.75"),
        "risk_score": 0.81,
        "application_date": date(2026, 8, 21),
        "event_timestamp": datetime(
            2026,
            8,
            21,
            9,
            45,
            0,
        ),
        "tags": [
            "broker",
            "manual-review",
        ],
        "attributes": {
            "channel": "broker",
            "campaign": "spring",
        },
        "address": {
            "suburb": "North Sydney",
            "postcode": "2060",
        },
        "events": [
            {
                "event_type": "created",
                "event_value": 1.0,
            },
            {
                "event_type": "approved",
                "event_value": 0.9,
            },
        ],
    },
    {
        "customer_id": 1005,
        "email_address": "diana@example.com",
        "is_active": False,
        "amount_financed": Decimal("4200.00"),
        "risk_score": 0.42,
        "application_date": date(2026, 8, 22),
        "event_timestamp": datetime(
            2026,
            8,
            22,
            14,
            20,
            0,
        ),
        "tags": [],
        "attributes": {},
        "address": {
            "suburb": "Melbourne",
            "postcode": "3000",
        },
        "events": [
            {
                "event_type": "created",
                "event_value": 1.0,
            },
            {
                "event_type": "declined",
                "event_value": 0.0,
            },
        ],
    },
]
sample_dataframe = spark.createDataFrame(
    sample_data,
    schema = sample_schema,
)
print("DataFrame type:", type(sample_dataframe))
print("Column count from native schema:", len(sample_dataframe.schema.fields))

# Backend
backend_identity = identify_backend(sample_dataframe)
print("BACKEND")
print(json.dumps(backend_identity.as_dict(), indent=2))
assert backend_identity.name is BackendName.PYSPARK
assert backend_identity.dataframe_api is DataFrameApi.SPARK_SQL
assert backend_identity.frame_kind is FrameKind.DATAFRAME
assert backend_identity.evaluation_mode is EvaluationMode.LAZY
assert backend_identity.distribution_mode is DistributionMode.DISTRIBUTED
assert backend_identity.connection_mode in {
    ConnectionMode.SPARK_CLASSIC,
    ConnectionMode.SPARK_CONNECT,
}
dataframe_input = resolve_dataframe_input(sample_dataframe)
assert dataframe_input.dataframe is sample_dataframe
assert dataframe_input.backend == backend_identity
assert "alice@example.com" not in repr(dataframe_input)

# Metadata-only discovery reads the `StructType` but does not call `count()`
metadata_only_schema = discover_physical_schema_from_input(
    dataframe_input,
    row_count_mode = RowCountMode.METADATA_ONLY,
)
assert metadata_only_schema.backend is dataframe_input.backend
print("\nMETADATA-ONLY DIMENSIONS")
print("shape:", metadata_only_schema.shape)
print("row count:", metadata_only_schema.dimensions.row_count)
print("column count:", metadata_only_schema.dimensions.column_count)
print(
    "row-count source:",
    metadata_only_schema.dimensions.row_count_source.value,
)
print(
    "has exact row count:",
    metadata_only_schema.dimensions.has_exact_row_count,
)

assert metadata_only_schema.shape == (None, len(sample_schema.fields))
assert (
    metadata_only_schema.dimensions.row_count_source
    is RowCountSource.NOT_COMPUTED
)
assert not metadata_only_schema.dimensions.has_exact_row_count
print("\nCOLUMNS")
for column in metadata_only_schema.columns:
    print(
        {
            "position": column.position,
            "exact_name_object": column.name,
            "report_name": column.name_string,
            "native_dtype_object": column.native_data_type,
            "native_dtype_string": column.native_data_type_string,
            "native_dtype_class": column.native_data_type_class,
            "nullable_schema_metadata": column.nullable,
        }
    )

# The complete native Spark schema is retained as a `StructType`
assert isinstance(metadata_only_schema.native_schema, StructType)
assert metadata_only_schema.native_schema == sample_dataframe.schema
assert metadata_only_schema.native_schema == sample_schema
print("\nNATIVE SPARK SCHEMA")
print(metadata_only_schema.native_schema.simpleString())
sample_dataframe.printSchema()
# Top-level nullability is copied from each `StructField`
expected_nullability = [field.nullable for field in sample_schema.fields]
discovered_nullability = [
    column.nullable for column in metadata_only_schema.columns
]
assert discovered_nullability == expected_nullability
# Parameterised and nested native type objects are retained intact
columns_by_name = {
    column.name_string: column for column in metadata_only_schema.columns
}
assert isinstance(
    columns_by_name["amount_financed"].native_data_type,
    DecimalType,
)
assert columns_by_name["amount_financed"].native_data_type.precision == 12
assert columns_by_name["amount_financed"].native_data_type.scale == 2
tags_data_type = columns_by_name["tags"].native_data_type
assert isinstance(tags_data_type, ArrayType)
assert tags_data_type.containsNull is False
attributes_data_type = columns_by_name["attributes"].native_data_type
assert isinstance(attributes_data_type, MapType)
assert attributes_data_type.valueContainsNull is True
address_data_type = columns_by_name["address"].native_data_type
assert isinstance(address_data_type, StructType)
assert address_data_type["suburb"].nullable is False
assert address_data_type["postcode"].nullable is True
events_data_type = columns_by_name["events"].native_data_type
assert isinstance(events_data_type, ArrayType)
assert events_data_type.containsNull is False
assert isinstance(events_data_type.elementType, StructType)
assert events_data_type.elementType["event_type"].nullable is False
# `StructField` metadata remains available through the complete native schema
identifier_metadata = metadata_only_schema.native_schema[
    "customer_id"
].metadata
assert identifier_metadata["primary_key_candidate"] is True

# `as_dict` removes backend-native objects and is JSON serialisable
report_dictionary = metadata_only_schema.as_dict()
report_json = json.dumps(report_dictionary, indent = 2)
print("\nJSON-SAFE METADATA-ONLY REPORT")
print(report_json)
convenience_schema = discover_physical_schema(sample_dataframe)
assert convenience_schema.as_dict() == metadata_only_schema.as_dict()
# `EXACT` executes `DataFrame.count()` for a bounded Spark dataframe
exact_schema = discover_physical_schema(
    sample_dataframe,
    row_count_mode = RowCountMode.EXACT,
)
assert exact_schema.shape == (5, len(sample_schema.fields))
assert exact_schema.dimensions.row_count == 5
assert (
    exact_schema.dimensions.row_count_source
    is RowCountSource.EXECUTED_QUERY
)
assert exact_schema.dimensions.has_exact_row_count

print("\nEXACT DIMENSIONS")
print(exact_schema.dimensions.as_dict())

# No row values are included in representations intended for diagnostics
assert "alice@example.com" not in repr(dataframe_input)
assert "alice@example.com" not in repr(metadata_only_schema)

Spark session started with version number 4.2.0
DataFrame type: <class 'pyspark.sql.classic.dataframe.DataFrame'>
Column count from native schema: 11
BACKEND
{
  "name": "pyspark",
  "dataframe_api": "spark_sql",
  "frame_kind": "dataframe",
  "evaluation_mode": "lazy",
  "distribution_mode": "distributed",
  "connection_mode": "spark_classic",
  "dataframe_type": "pyspark.sql.classic.dataframe.DataFrame",
  "client_library_version": "4.2.0"
}

METADATA-ONLY DIMENSIONS
shape: (None, 11)
row count: None
column count: 11
row-count source: not_computed
has exact row count: False

COLUMNS
{'position': 0, 'exact_name_object': 'customer_id', 'report_name': 'customer_id', 'native_dtype_object': LongType(), 'native_dtype_string': 'bigint', 'native_dtype_class': 'pyspark.sql.types.LongType', 'nullable_schema_metadata': False}
{'position': 1, 'exact_name_object': 'email_address', 'report_name': 'email_address', 'native_dtype_object': StringType(), 'native_dtype_string': 'string', 'native_dtype_c

##### Execptions

In [8]:
# Expected public input/type errors
try:
    identify_backend(None)
except MissingDataFrameError as error:
    print("\nEXPECTED MISSING INPUT ERROR:", error)

try:
    identify_backend(Row(customer_identifier = 1))
except UnsupportedDataFrameError as error:
    print("EXPECTED UNSUPPORTED INPUT ERROR:", error)

try:
    discover_physical_schema_from_input(
        sample_dataframe
    )
except TypeError as error:
    print("EXPECTED RAW-INPUT ERROR:", error)

try:
    discover_physical_schema_from_input(
        dataframe_input,
        row_count_mode = "exact"
    )
except TypeError as error:
    print("EXPECTED ROW-COUNT MODE ERROR:", error)

# Backend detector
class NeverMatchingDetector:
    name = "example-never-matching"
    supported_inputs = ("example.NeverMatchingDataFrame",)

    def detect(self, 
               dataframe: object) -> BackendIdentity | None:
        
        return None
extended_backend_registry = DEFAULT_BACKEND_REGISTRY.with_detector(
    NeverMatchingDetector()
)
assert len(extended_backend_registry.detectors) == (
    len(DEFAULT_BACKEND_REGISTRY.detectors) + 1
)
assert len(DEFAULT_BACKEND_REGISTRY.detectors) == 3
assert (
    identify_backend(
        sample_dataframe,
        registry = extended_backend_registry,
    ).name
    is BackendName.PYSPARK
)
print("\nSUPPORTED INPUTS")
print(DEFAULT_BACKEND_REGISTRY.supported_inputs)

# Discoverer
default_pyspark_discoverer = next(
    discoverer
    for discoverer in DEFAULT_PHYSICAL_SCHEMA_REGISTRY.discoverers
    if discoverer.backend_name is BackendName.PYSPARK
)
class RecordingPySparkSchemaDiscoverer:
    backend_name = BackendName.PYSPARK

    def __init__(self, 
                 delegate: PhysicalSchemaDiscoverer) -> None:
        self.delegate = delegate
        self.call_count = 0

    def discover(self,
                 dataframe: object,
                 backend: BackendIdentity,
                 row_count_mode: RowCountMode) -> DataFramePhysicalSchema:
        self.call_count += 1

        return self.delegate.discover(
            dataframe,
            backend,
            row_count_mode,
        )
recording_discoverer = RecordingPySparkSchemaDiscoverer(
    default_pyspark_discoverer
)
empty_schema_registry = PhysicalSchemaRegistry(())
recording_schema_registry = empty_schema_registry.with_discoverer(
    recording_discoverer
)
assert empty_schema_registry.discoverers == ()
recorded_schema = discover_physical_schema(
    sample_dataframe,
    schema_registry = recording_schema_registry,
)
assert recorded_schema.shape == (None, len(sample_schema.fields))
assert recording_discoverer.call_count == 1

# Registry construction rejects duplicates
try:
    BackendRegistry((NeverMatchingDetector(), NeverMatchingDetector()))
except DuplicateDetectorNameError as error:
    print("EXPECTED DUPLICATE DETECTOR ERROR:", error)
try:
    PhysicalSchemaRegistry(
        (default_pyspark_discoverer, default_pyspark_discoverer)
    )
except DuplicateSchemaDiscovererError as error:
    print("EXPECTED DUPLICATE DISCOVERER ERROR:", error)

# If two detectors claim the same object, resolution fails explicitly
class ConflictingPySparkDetector:
    name = "conflicting-pyspark"
    supported_inputs = ("pyspark.sql.DataFrame",)

    def detect(self, 
               dataframe: object) -> BackendIdentity | None:
        if dataframe is sample_dataframe:

            return backend_identity

        return None
ambiguous_registry = DEFAULT_BACKEND_REGISTRY.with_detector(
    ConflictingPySparkDetector()
)
try:
    identify_backend(sample_dataframe, registry = ambiguous_registry)
except AmbiguousBackendError as error:
    print("EXPECTED AMBIGUOUS BACKEND ERROR:", error)

# Unexpected detector failures are wrapped and retain their original cause
class FailingBackendDetector:
    name = "failing"
    supported_inputs = ("pyspark.sql.DataFrame",)

    def detect(self, 
               dataframe: object) -> BackendIdentity | None:
        raise RuntimeError("Underlying detector failure")
try:
    identify_backend(
        sample_dataframe,
        registry = BackendRegistry((FailingBackendDetector(),)),
    )
except BackendDetectorError as error:
    assert isinstance(error.__cause__, RuntimeError)
    print("EXPECTED WRAPPED DETECTOR ERROR:", error)

# A resolved backend needs a corresponding physical-schema discoverer.
try:
    discover_physical_schema_from_input(
        dataframe_input,
        registry = PhysicalSchemaRegistry(()),
    )
except UnsupportedSchemaDiscoveryBackendError as error:
    print("EXPECTED MISSING DISCOVERER ERROR:", error)

# Unexpected schema-discovery failures are likewise wrapped and chained.
class FailingPySparkSchemaDiscoverer:
    backend_name = BackendName.PYSPARK

    def discover(self,
                 dataframe: object,
                 backend: BackendIdentity,
                 row_count_mode: RowCountMode) -> DataFramePhysicalSchema:
        raise RuntimeError("Underlying schema-discovery failure")
try:
    discover_physical_schema_from_input(
        dataframe_input,
        registry = PhysicalSchemaRegistry(
            (FailingPySparkSchemaDiscoverer(),)
        ),
    )
except PhysicalSchemaDiscoveryError as error:
    assert isinstance(error.__cause__, RuntimeError)
    print("EXPECTED WRAPPED SCHEMA ERROR:", error)


EXPECTED MISSING INPUT ERROR: Expected a dataframe but received `None`
EXPECTED UNSUPPORTED INPUT ERROR: Unsupported dataframe type 'pyspark.sql.types.Row', supported inputs are: pandas.DataFrame, polars.DataFrame, polars.LazyFrame, pyspark.sql.DataFrame (Spark classic), pyspark.sql.DataFrame (Spark Connect)
EXPECTED RAW-INPUT ERROR: Expected `DataFrameInput`, call `resolve_dataframe_input()` first or pass the native dataframe to `discover_physical_schema()`
EXPECTED ROW-COUNT MODE ERROR: row_count_mode must be a RowCountMode value.

SUPPORTED INPUTS
('pandas.DataFrame', 'polars.DataFrame', 'polars.LazyFrame', 'pyspark.sql.DataFrame (Spark classic)', 'pyspark.sql.DataFrame (Spark Connect)')
EXPECTED DUPLICATE DETECTOR ERROR: A backend detector named 'example-never-matching' is already registered
EXPECTED DUPLICATE DISCOVERER ERROR: A physical-schema discoverer for backend 'pyspark' is already registered
EXPECTED AMBIGUOUS BACKEND ERROR: Backend detection is ambiguous for 'pyspark.sql.